In [7]:
from pathlib import Path

import pandas as pd

SILVER_DEP_PATH = Path("../../data/silver/silver_DEP-departementale.csv")
GOLD_DIR = Path("../../data/gold/security")

print(f"Chargement du fichier : {SILVER_DEP_PATH}")
df_dep = pd.read_csv(SILVER_DEP_PATH, sep=";", dtype=str)

display(df_dep.head())
print(df_dep.shape)

Chargement du fichier : ../../data/silver/silver_DEP-departementale.csv


,Code_departement,Code_region,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime,extraction_source_url,ingestion_timestamp,source_file_name
0,69,84,2016,Homicides,Victime,23,0.0125279,1835903,2016,904462,2016,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146,DEP-departementale.csv
1,69,84,2017,Homicides,Victime,20,0.01085,1843319,2017,914980,2017,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146,DEP-departementale.csv
2,69,84,2018,Homicides,Victime,19,0.0102177,1859524,2018,926576,2018,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146,DEP-departementale.csv
3,69,84,2019,Homicides,Victime,21,0.0111955,1875747,2019,937544,2019,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146,DEP-departementale.csv
4,69,84,2020,Homicides,Victime,21,0.0111498,1883437,2020,949531,2020,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146,DEP-departementale.csv


(180, 14)


In [8]:
# Normalisation des cles et des types pour alignement avec le modele Gold existant

df_dep = df_dep.copy()

df_dep["Code_departement"] = df_dep["Code_departement"].astype(str).str.strip().str.zfill(2)
df_dep["Code_region"] = df_dep["Code_region"].astype(str).str.strip().str.zfill(2)

df_dep["annee"] = pd.to_numeric(df_dep["annee"], errors="coerce").astype("Int64")
df_dep["nombre"] = pd.to_numeric(df_dep["nombre"], errors="coerce")
df_dep["taux_pour_mille"] = pd.to_numeric(df_dep["taux_pour_mille"], errors="coerce")
df_dep["insee_pop"] = pd.to_numeric(df_dep["insee_pop"], errors="coerce").astype("Int64")
df_dep["insee_log"] = pd.to_numeric(df_dep["insee_log"], errors="coerce").astype("Int64")

df_dep["ingestion_timestamp"] = pd.to_datetime(df_dep["ingestion_timestamp"], errors="coerce")

df_dep["indicateur"] = df_dep["indicateur"].astype(str).str.strip()
df_dep["unite_de_compte"] = df_dep["unite_de_compte"].astype(str).str.strip()
df_dep["extraction_source_url"] = df_dep["extraction_source_url"].astype(str).str.strip()
df_dep["source_file_name"] = df_dep["source_file_name"].astype(str).str.strip()

# Création de l'id_departement
df_dep["id_departement"] = df_dep["Code_departement"]

display(df_dep.head())
print(df_dep.dtypes)

,Code_departement,Code_region,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime,extraction_source_url,ingestion_timestamp,source_file_name,id_departement
0,69,84,2016,Homicides,Victime,23,0.012528,1835903,2016,904462,2016,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146,DEP-departementale.csv,69
1,69,84,2017,Homicides,Victime,20,0.010850,1843319,2017,914980,2017,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146,DEP-departementale.csv,69
2,69,84,2018,Homicides,Victime,19,0.010218,1859524,2018,926576,2018,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146,DEP-departementale.csv,69
3,69,84,2019,Homicides,Victime,21,0.011196,1875747,2019,937544,2019,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146,DEP-departementale.csv,69
4,69,84,2020,Homicides,Victime,21,0.011150,1883437,2020,949531,2020,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146,DEP-departementale.csv,69


Code_departement                 object
Code_region                      object
annee                             Int64
indicateur                       object
unite_de_compte                  object
nombre                            int64
taux_pour_mille                 float64
insee_pop                         Int64
insee_pop_millesime              object
insee_log                         Int64
insee_log_millesime              object
extraction_source_url            object
ingestion_timestamp      datetime64[ns]
source_file_name                 object
id_departement                   object
dtype: object


In [9]:
# Construction de la dimension des indicateurs de securite départementaux

dim_indicateur_securite_dep = (
    df_dep[["indicateur", "unite_de_compte"]]
    .drop_duplicates()
    .sort_values(["indicateur", "unite_de_compte"])
    .reset_index(drop=True)
)

dim_indicateur_securite_dep["id_indicateur_securite"] = [
    f"SEC_DEP_{i:03d}" for i in range(1, len(dim_indicateur_securite_dep) + 1)
]

dim_indicateur_securite_dep = dim_indicateur_securite_dep[[
    "id_indicateur_securite",
    "indicateur",
    "unite_de_compte",
]]

display(dim_indicateur_securite_dep.head())
print(dim_indicateur_securite_dep.shape)

,id_indicateur_securite,indicateur,unite_de_compte
0,SEC_DEP_001,Cambriolages de logement,Infraction
1,SEC_DEP_002,Destructions et dégradations volontaires,Infraction
2,SEC_DEP_003,Escroqueries et fraudes aux moyens de paiement,Victime
3,SEC_DEP_004,Homicides,Victime
4,SEC_DEP_005,Tentatives d'homicide,Victime


(18, 3)


In [10]:
# Extraction des informations géographiques (Dimension Département)

cols_geo_dep = ["Code_departement", "Code_region"]
dim_departement = (
    df_dep[cols_geo_dep]
    .drop_duplicates()
    .rename(columns={"Code_departement": "id_departement", "Code_region": "code_region"})
    .reset_index(drop=True)
)

print(f"Taille Dim_Departement : {dim_departement.shape}")
display(dim_departement.head())

Taille Dim_Departement : (1, 2)


,id_departement,code_region
0,69,84


In [11]:
# Construction des tables de faits départementales
# SPLIT pour éviter la duplication des statistiques démographiques par indicateur !

fact_df_full = df_dep.merge(
    dim_indicateur_securite_dep,
    on=["indicateur", "unite_de_compte"],
    how="left"
)

# 1. Table de faits : Démographie/Logement Département (Grain = id_departement + annee)
fact_demographie_dep = fact_df_full[[
    "id_departement",
    "annee",
    "insee_pop",
    "insee_log",
]].drop_duplicates().copy()

fact_demographie_dep = fact_demographie_dep.sort_values([
    "id_departement",
    "annee"
]).reset_index(drop=True)

print(f"Taille Fact_Demographie_Dep : {fact_demographie_dep.shape}")
display(fact_demographie_dep.head())


# 2. Table de faits : Sécurité Départementale (Grain = id_departement + annee + id_indicateur_securite)
fact_securite_dep = fact_df_full[[
    "id_departement",
    "annee",
    "id_indicateur_securite",
    "nombre",
    "taux_pour_mille",
    "source_file_name",
    "extraction_source_url",
    "ingestion_timestamp",
]].copy()

fact_securite_dep = fact_securite_dep.sort_values([
    "id_departement",
    "annee",
    "id_indicateur_securite",
]).reset_index(drop=True)

print(f"Taille Fact_Securite_Dep : {fact_securite_dep.shape}")
display(fact_securite_dep.head())

Taille Fact_Demographie_Dep : (10, 4)


,id_departement,annee,insee_pop,insee_log
0,69,2016,1835903,904462
1,69,2017,1843319,914980
2,69,2018,1859524,926576
3,69,2019,1875747,937544
4,69,2020,1883437,949531


Taille Fact_Securite_Dep : (180, 8)


,id_departement,annee,id_indicateur_securite,nombre,taux_pour_mille,source_file_name,extraction_source_url,ingestion_timestamp
0,69,2016,SEC_DEP_001,10234,11.315014,DEP-departementale.csv,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146
1,69,2016,SEC_DEP_002,18847,10.265793,DEP-departementale.csv,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146
2,69,2016,SEC_DEP_003,9305,5.068351,DEP-departementale.csv,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146
3,69,2016,SEC_DEP_004,23,0.012528,DEP-departementale.csv,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146
4,69,2016,SEC_DEP_005,60,0.032682,DEP-departementale.csv,https://www.data.gouv.fr/api/1/datasets/r/2b27...,2026-03-28 19:12:32.815146


In [12]:
# Sauvegarde des tables Gold securite départementale

GOLD_DIR.mkdir(parents=True, exist_ok=True)

dim_ind_output_path = GOLD_DIR / "dim_indicateur_securite_dep.csv"
dim_dep_output_path = GOLD_DIR / "dim_departement.csv"
fact_demo_output_path = GOLD_DIR / "fact_demographie_dep.csv"
fact_sec_output_path = GOLD_DIR / "fact_securite_dep.csv"

# Sauvegarde Dimensions
dim_indicateur_securite_dep.to_csv(dim_ind_output_path, sep=";", index=False)
dim_departement.to_csv(dim_dep_output_path, sep=";", index=False)
print(f"Dimensions sauvegardées : {dim_ind_output_path} & {dim_dep_output_path}")

# Sauvegarde Faits (Sécurité + Démographie/Logement)
fact_demographie_dep.to_csv(fact_demo_output_path, sep=";", index=False)
fact_securite_dep.to_csv(fact_sec_output_path, sep=";", index=False)

print(f"Fact Demographie Dep sauvegardée : {fact_demo_output_path}")
print(f"Fact Securite Dep sauvegardée : {fact_sec_output_path}")
print("✅ Modèle en étoile départemental généré avec succès (grain épuré)")

Dimensions sauvegardées : ../../data/gold/security/dim_indicateur_securite_dep.csv & ../../data/gold/security/dim_departement.csv
Fact Demographie Dep sauvegardée : ../../data/gold/security/fact_demographie_dep.csv
Fact Securite Dep sauvegardée : ../../data/gold/security/fact_securite_dep.csv
✅ Modèle en étoile départemental généré avec succès (grain épuré)
